<a href="https://colab.research.google.com/github/RaJ-0002/DataScience/blob/master/Final_assignment_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# !pip install schedule

In [10]:
# pip install pandas requests tqdm

In [11]:
# !pip install yfinance

In [12]:
import pandas as pd
import requests
import io
import numpy as np
import os
from tqdm import tqdm
from datetime import datetime, timedelta
import requests, io, zipfile, datetime, time
import schedule
# import time
import yfinance as yf

In [13]:
import yfinance as yf
import pandas as pd

# -----------------------------
# Download Historical Data
# -----------------------------
nasdaq = yf.download("^IXIC", start="2024-01-01", end="2024-01-31")
gold = yf.download("GC=F", start="2024-01-01", end="2024-01-31")
btc = yf.download("BTC-USD", start="2024-01-01", end="2024-01-31")

# -----------------------------
# Combine 'Close' Prices
# -----------------------------
data = pd.concat([nasdaq['Close'], gold['Close'], btc['Close']], axis=1)
data.columns = ['NASDAQ', 'Gold', 'Bitcoin']

# Reset the index so 'Date' becomes a column instead of index
data.reset_index(inplace=True)

# -----------------------------
# Save to CSV File
# -----------------------------
data.to_csv("btc.csv", index=False)

print("✅ File saved successfully as 'Market_Prices_2024.csv'")
print(data.head())


/tmp/ipython-input-3840337478.py:7: FutureWarning: YF.download() has changed argument auto_adjust default to True
  nasdaq = yf.download("^IXIC", start="2024-01-01", end="2024-01-31")
[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-3840337478.py:8: FutureWarning: YF.download() has changed argument auto_adjust default to True
  gold = yf.download("GC=F", start="2024-01-01", end="2024-01-31")
[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-3840337478.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  btc = yf.download("BTC-USD", start="2024-01-01", end="2024-01-31")
[*********************100%***********************]  1 of 1 completed

✅ File saved successfully as 'Market_Prices_2024.csv'
        Date        NASDAQ         Gold       Bitcoin
0 2024-01-01           NaN          NaN  44167.332031
1 2024-01-02  14765.940430  2064.399902  44957.968750
2 2024-01-03  14592.209961  2034.199951  42848.175781
3 2024-01-04  14510.299805  2042.300049  44179.921875
4 2024-01-05  14524.070312  2042.400024  44162.691406


In [14]:
# ===================================================>v5
import pandas as pd
import requests, io, zipfile, time
from tqdm import tqdm
from datetime import datetime


BASE_URL = "http://data.gdeltproject.org/gdeltv2/"
MAX_RETRIES = 3
RETRY_DELAY = 2
SLEEP_BETWEEN = 0.25
TIMEOUT = 60


# GDELT 2.0 Event Export Column Names (complete list - 58 columns)
GDELT_V2_EXPORT_COLUMNS = [
    "GLOBALEVENTID", "SQLDATE", "MonthYear", "Year", "FractionDate",

    "Actor1Code", "Actor1Name", "Actor1CountryCode", "Actor1KnownGroupCode",
    "Actor1EthnicCode", "Actor1Religion1Code", "Actor1Religion2Code",
    "Actor1Type1Code", "Actor1Type2Code", "Actor1Type3Code",

    "Actor2Code", "Actor2Name", "Actor2CountryCode", "Actor2KnownGroupCode",
    "Actor2EthnicCode", "Actor2Religion1Code", "Actor2Religion2Code",
    "Actor2Type1Code", "Actor2Type2Code", "Actor2Type3Code",

    "IsRootEvent", "EventCode", "EventBaseCode", "EventRootCode",
    "QuadClass", "GoldsteinScale", "NumMentions", "NumSources", "NumArticles",
    "AvgTone",

    "Actor1Geo_Type", "Actor1Geo_FullName", "Actor1Geo_CountryCode",
    "Actor1Geo_ADM1Code", "Actor1Geo_ADM2Code", "Actor1Geo_Lat", "Actor1Geo_Long",
    "Actor1Geo_FeatureID",

    "Actor2Geo_Type", "Actor2Geo_FullName",
    "Actor2Geo_CountryCode", "Actor2Geo_ADM1Code", "Actor2Geo_ADM2Code",
    "Actor2Geo_Lat", "Actor2Geo_Long", "Actor2Geo_FeatureID",

    "ActionGeo_Type", "ActionGeo_FullName", "ActionGeo_CountryCode",
    "ActionGeo_ADM1Code", "ActionGeo_ADM2Code", "ActionGeo_Lat",
    "ActionGeo_Long", "ActionGeo_FeatureID", "DATEADDED", "SOURCEURL"
]


def _download_with_retry(url):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = requests.get(url, timeout=TIMEOUT)
            if r.status_code == 200:
                return r.content
            elif r.status_code == 404:
                print(f"⚠️  Not found: {url}")
                return None
            else:
                print(f"⚠️  HTTP {r.status_code} on attempt {attempt}")
        except Exception as e:
            print(f"⚠️  Error attempt {attempt}: {e}")
        time.sleep(RETRY_DELAY)
    return None


def fetch_gdelt_v2_events_by_event_date(event_start_date, event_end_date, frequency='D'):
    """
    Downloads GDELT 2.0 events that OCCURRED between event_start_date and event_end_date.

    NOTE: GDELT files are organized by DATEADDED (when processed), not SQLDATE (when event occurred).
    This function downloads files from the corresponding period and filters by actual event date (SQLDATE).

    For events in Jan-Mar 2024, we download files from Jan-Mar 2024 (most events are added same day).
    """
    all_data = []
    # Download files from the same date range (most events are added on the day they occur)
    all_dates = pd.date_range(event_start_date, event_end_date, freq=frequency)

    print(f"📥 Downloading files from {event_start_date} to {event_end_date}")
    print(f"🎯 Will filter for events that occurred (SQLDATE) in this period\n")

    for dt in tqdm(all_dates, desc="Downloading GDELT 2.0 events"):
        ts = dt.strftime("%Y%m%d000000")
        url = f"{BASE_URL}{ts}.export.CSV.zip"

        content = _download_with_retry(url)
        if not content:
            continue

        try:
            with zipfile.ZipFile(io.BytesIO(content)) as zf:
                name = zf.namelist()[0]
                with zf.open(name) as f:
                    df = pd.read_csv(
                        f,
                        sep="\t",
                        header=None,
                        low_memory=False,
                        names=GDELT_V2_EXPORT_COLUMNS
                    )
                    df["_source_date"] = dt.strftime("%Y%m%d")
                    df["_source_file"] = name
                    all_data.append(df)
        except Exception as e:
            print(f"❌ Error processing {ts}: {e}")
            continue

        time.sleep(SLEEP_BETWEEN)

    if not all_data:
        raise ValueError("No data downloaded — check date range or connectivity.")

    result = pd.concat(all_data, ignore_index=True)

    # CRITICAL: Filter by actual event date (SQLDATE), not when it was added (DATEADDED)
    start_sqldate = int(event_start_date.replace("-", ""))
    end_sqldate = int(event_end_date.replace("-", ""))

    print(f"\n📊 Total rows before filtering: {len(result)}")

    # Convert SQLDATE to int for comparison (keep raw data otherwise)
    result["SQLDATE"] = pd.to_numeric(result["SQLDATE"], errors="coerce")

    # Filter for events that actually occurred in our target date range
    result = result[(result["SQLDATE"] >= start_sqldate) & (result["SQLDATE"] <= end_sqldate)]

    print(f"✅ Rows after filtering by event date (SQLDATE): {len(result)}")
    print(f"📅 Event date range in data: {result['SQLDATE'].min()} to {result['SQLDATE'].max()}")

    return result


# Download events that OCCURRED in January to March 2024
if __name__ == "__main__":
    df = fetch_gdelt_v2_events_by_event_date(
        event_start_date="2024-01-01",
        event_end_date="2024-01-31",  # Include full March
        frequency='D'
    )

    print("\n" + "="*80)
    print(df.head(10))
    print("\n✅ Total rows:", len(df))
    print("\nShape:", df.shape)
    print("\nColumn names:")
    print(df.columns.tolist())

    # Verify the data
    print("\n🔍 Data Verification:")
    print(f"   SQLDATE range: {df['SQLDATE'].min()} to {df['SQLDATE'].max()}")
    print(f"   Unique event dates: {df['SQLDATE'].nunique()}")

    df.to_csv("gdelt_v2_events_JAN_2024_by_event_date.csv", index=False)
    print("\n✅ Data saved to: gdelt_v2_events_JAN_2024_by_event_date.csv")

📥 Downloading files from 2024-01-01 to 2024-01-31
🎯 Will filter for events that occurred (SQLDATE) in this period




📊 Total rows before filtering: 87623
✅ Rows after filtering by event date (SQLDATE): 86202
📅 Event date range in data: 20240101 to 20240131

    GLOBALEVENTID   SQLDATE  MonthYear  Year  FractionDate Actor1Code  \
22     1149157574  20240101     202401  2024     2024.0027        NaN   
23     1149157575  20240101     202401  2024     2024.0027        NaN   
24     1149157576  20240101     202401  2024     2024.0027        NaN   
25     1149157577  20240101     202401  2024     2024.0027        NaN   
26     1149157578  20240101     202401  2024     2024.0027        NaN   
27     1149157579  20240101     202401  2024     2024.0027        NaN   
28     1149157580  20240101     202401  2024     2024.0027        NaN   
29     1149157581  20240101     202401  2024     2024.0027        NaN   
30     1149157582  20240101     202401  2024     2024.0027        NaN   
31     1149157583  20240101     202401  2024     2024.0027        NaN   

   Actor1Name Actor1CountryCode Actor1KnownGroupCode A

In [15]:
import pandas as pd
import requests, io, zipfile, time
from tqdm import tqdm
from datetime import datetime


def fetch_gdelt_mentions_sampled(start_date, end_date, frequency='D'):
    """
    Efficiently downloads GDELT Mentions files between start_date and end_date.
    Corrected to match the official 16-column schema.
    """
    base_url = "http://data.gdeltproject.org/gdeltv2/"
    dates = pd.date_range(start_date, end_date, freq=frequency)
    all_data = []

    colnames = [
        "GlobalEventID", "EventTimeDate", "MentionTimeDate", "MentionType",
        "MentionSourceName", "MentionIdentifier", "SentenceID",
        "Actor1CharOffset", "Actor2CharOffset", "ActionCharOffset",
        "InRawText", "Confidence", "MentionDocTone",
        "MentionDocTranslationInfo", "Extras1", "Extras2"
    ]

    for dt in tqdm(dates, desc="Fetching Mentions Files"):
        timestamp = dt.strftime("%Y%m%d000000")  # Midnight UTC per day
        url = f"{base_url}{timestamp}.mentions.CSV.zip"

        try:
            r = requests.get(url, timeout=30)
            if r.status_code != 200:
                print(f"⚠️ Skipped {timestamp}: HTTP {r.status_code}")
                continue

            with zipfile.ZipFile(io.BytesIO(r.content)) as z:
                filename = z.namelist()[0]
                with z.open(filename) as f:
                    df = pd.read_csv(
                        f,
                        sep="\t",
                        header=None,
                        names=colnames,
                        dtype=str,
                        engine="python",
                        quoting=3,   # ignore quotes
                        on_bad_lines="skip"
                    )

            if not df.empty:
                all_data.append(df)

        except Exception as e:
            print(f"❌ Error fetching {timestamp}: {e}")

        time.sleep(0.2)  # Respectful pause between requests

    if not all_data:
        raise ValueError("No valid Mentions data downloaded.")

    mentions_df = pd.concat(all_data, ignore_index=True)

    # Convert date columns safely
    for col in ["EventTimeDate", "MentionTimeDate"]:
        mentions_df[col] = pd.to_datetime(mentions_df[col], errors='coerce', format='%Y%m%d%H%M%S')

    return mentions_df


# Example: Fetch one file per day for January 2024
mentions_df = fetch_gdelt_mentions_sampled("2024-01-01", "2024-01-31")

print("✅ Total mentions:", len(mentions_df))
print("\n📄 Sample Data:")
print(mentions_df.head(3))
print("\nData types:")
print(mentions_df.dtypes)


Fetching Mentions Files: 100%|██████████| 31/31 [00:08<00:00,  3.54it/s]

✅ Total mentions: 126525

📄 Sample Data:
  GlobalEventID EventTimeDate MentionTimeDate MentionType MentionSourceName  \
0    1149157552    2024-01-01      2024-01-01           1      theblaze.com   
1    1149157553    2024-01-01      2024-01-01           1         yahoo.com   
2    1149157554    2024-01-01      2024-01-01           1    nichegamer.com   

                                   MentionIdentifier SentenceID  \
0  https://www.theblaze.com/columns/opinion/roth-...         10   
1  https://news.yahoo.com/china-xi-vows-reunifica...          2   
2  https://nichegamer.com/future-boy-conan-gettin...          1   

  Actor1CharOffset Actor2CharOffset ActionCharOffset InRawText Confidence  \
0               -1             3181             3140         1        100   
1              923              933              943         0         20   
2               68               -1              137         0         20   

  MentionDocTone MentionDocTranslationInfo Extras1 Extras2  
0  

In [17]:
import pandas as pd
import requests
import io
import zipfile
import time
from tqdm import tqdm


# GKG 2.1 column definitions (verified)
GKG_V21_COLUMNS = [
    "GKGRECORDID", "V2.1DATE", "V2SOURCECOLLECTIONIDENTIFIER", "V2SOURCECOMMONNAME",
    "V2DOCUMENTIDENTIFIER", "V1COUNTS", "V2.1COUNTS", "V1THEMES", "V2ENHANCEDTHEMES",
    "V1LOCATIONS", "V2ENHANCEDLOCATIONS", "V1PERSONS", "V2ENHANCEDPERSONS",
    "V1ORGANIZATIONS", "V2ENHANCEDORGANIZATIONS", "V1.5TONE", "V2.1ENHANCEDDATES",
    "V2GCAM", "V2.1SHARINGIMAGE", "V2.1RELATEDIMAGES", "V2.1SOCIALIMAGEEMBEDS",
    "V2.1SOCIALVIDEOEMBEDS", "V2.1QUOTATIONS", "V2.1ALLNAMES", "V2.1AMOUNTS",
    "V2.1TRANSLATIONINFO", "V2EXTRASXML"
]


def fetch_gdelt_gkg_sampled(start_date, end_date, frequency='D'):
    """
    Efficiently downloads sampled GDELT GKG 2.1 files between start_date and end_date.
    Defaults to one file per day to reduce load.
    """
    base_url = "http://data.gdeltproject.org/gkg/"
    all_data = []
    dates = pd.date_range(start_date, end_date, freq=frequency)

    for dt in tqdm(dates, desc="Fetching GKG 2.1 Files"):
        timestamp = dt.strftime("%Y%m%d")
        url = f"{base_url}{timestamp}.gkg.csv.zip"

        try:
            r = requests.get(url, timeout=30)
            if r.status_code != 200:
                print(f"⚠️ Skipped {timestamp} (HTTP {r.status_code})")
                continue

            with zipfile.ZipFile(io.BytesIO(r.content)) as z:
                filename = z.namelist()[0]
                with z.open(filename) as f:
                    df = pd.read_csv(
                        f,
                        sep="\t",
                        header=None,
                        names=GKG_V21_COLUMNS,
                        dtype=str,
                        engine="python",
                        on_bad_lines="skip",
                        quoting=3
                    )

            if not df.empty:
                df["V2.1DATE"] = pd.to_datetime(df["V2.1DATE"], format="%Y%m%d%H%M%S", errors="coerce")
                all_data.append(df)

        except Exception as e:
            print(f"❌ Error fetching {timestamp}: {e}")

        time.sleep(0.2)  # Respectful pause

    if not all_data:
        raise ValueError("No GKG data downloaded.")

    gkg_df = pd.concat(all_data, ignore_index=True)
    return gkg_df


# Example: Fetch from January to March 2024
gkg_df = fetch_gdelt_gkg_sampled("2024-01-01", "2024-01-31", frequency='D')

print("✅ Total records:", len(gkg_df))
print("\n📄 Sample Data:")
print(gkg_df.head(3))
print("\nData types:")
print(gkg_df.dtypes)


Fetching GKG 2.1 Files: 100%|██████████| 31/31 [01:31<00:00,  2.95s/it]


✅ Total records: 2888992

📄 Sample Data:
  GKGRECORDID V2.1DATE                       V2SOURCECOLLECTIONIDENTIFIER  \
0        DATE      NaT                                             COUNTS   
1    20240101      NaT                                                NaN   
2    20240101      NaT  KILL#2##2#Colorado, United States#US#USCO#39.0...   

                                  V2SOURCECOMMONNAME  \
0                                             THEMES   
1  UNGP_FORESTS_RIVERS_OCEANS;WB_137_WATER;ECON_D...   
2  KILL;TAX_FNCACT;TAX_FNCACT_CHILDREN;WOUND;ARRE...   

                                V2DOCUMENTIDENTIFIER           V1COUNTS  \
0                                          LOCATIONS            PERSONS   
1  4#Mediterranean Sea, Oceans (General), Oceans#...       eric akopian   
2  4#London, London, City Of, United Kingdom#UK#U...  kimberlee singler   

                                          V2.1COUNTS  \
0                                      ORGANIZATIONS   
1  european 

In [18]:
gkg_df

,GKGRECORDID,V2.1DATE,V2SOURCECOLLECTIONIDENTIFIER,V2SOURCECOMMONNAME,V2DOCUMENTIDENTIFIER,V1COUNTS,V2.1COUNTS,V1THEMES,V2ENHANCEDTHEMES,V1LOCATIONS,...,V2GCAM,V2.1SHARINGIMAGE,V2.1RELATEDIMAGES,V2.1SOCIALIMAGEEMBEDS,V2.1SOCIALVIDEOEMBEDS,V2.1QUOTATIONS,V2.1ALLNAMES,V2.1AMOUNTS,V2.1TRANSLATIONINFO,V2EXTRASXML
0,DATE,NaT,COUNTS,THEMES,LOCATIONS,PERSONS,ORGANIZATIONS,TONE,CAMEOEVENTIDS,SOURCES,...,None,None,None,None,None,None,None,None,None,None
1,20240101,NaT,NaN,UNGP_FORESTS_RIVERS_OCEANS;WB_137_WATER;ECON_D...,"4#Mediterranean Sea, Oceans (General), Oceans#...",eric akopian,european commission;united nations environment...,"0.985221674876847,4.31034482758621,3.325123152...",NaN,euronews.com,...,None,None,None,None,None,None,None,None,None,None
2,20240101,NaT,"KILL#2##2#Colorado, United States#US#USCO#39.0...",KILL;TAX_FNCACT;TAX_FNCACT_CHILDREN;WOUND;ARRE...,"4#London, London, City Of, United Kingdom#UK#U...",kimberlee singler,westminster magistrate court;united states,"-9.5959595959596,1.01010101010101,10.606060606...","1149245378,1149245378,1149245378,1149245378,11...",fox23maine.com;katv.com;cbs6albany.com;wwmt.co...,...,None,None,None,None,None,None,None,None,None,None
3,20240101,NaT,"KILL#24##4#Gaza, Israel (General), Israel#IS#I...",ARMEDCONFLICT;EPU_CATS_NATIONAL_SECURITY;KILL;...,"1#West Bank#WE#WE#31.666667#35.25#WE;4#Gaza, I...",bassam hana;gabriel zemelman;itamar ben gvir;s...,united states;european union,"-7.80590717299578,1.58227848101266,9.388185654...","1149171681,1149164915,1149233587,1149182237,11...",lebanondemocrat.com,...,None,None,None,None,None,None,None,None,None,None
4,20240101,NaT,NaN,TAX_ETHNICITY;TAX_ETHNICITY_AUSTRALIAN;MEDIA_M...,"4#Mudgee, New South Wales, Australia#AS#AS02#-...",shelley craft;scott cam;peter druitt,first national,"-0.952380952380952,0.952380952380952,1.9047619...",1149164472,mudgeeguardian.com.au,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2888987,20240131,NaT,NaN,CRISISLEX_CRISISLEXREC;WB_1979_NATURAL_RESOURC...,"3#San Gabriel Mountains, California, United St...",los angeles,national weather service,"-2.23214285714286,0.892857142857143,3.125,4.01...",NaN,mynewsla.com,...,None,None,None,None,None,None,None,None,None,None
2888988,20240131,NaT,NaN,TAX_DISEASE;TAX_DISEASE_COVID;HEALTH_VACCINATI...,"2#Florida, United States#US#USFL#27.8333#-81.7...",mandy cohen;robert califf;joseph ladapo;jan je...,u s centers for disease;drug administration;co...,"-2.08667736757624,1.28410914927769,3.370786516...","1154876104,1154858113,1154811340,1154811416,11...",rightspeak.net,...,None,None,None,None,None,None,None,None,None,None
2888989,20240131,NaT,NaN,TAX_DISEASE;TAX_DISEASE_INFECTIOUS;GENERAL_HEA...,1#United Kingdom#UK#UK#54#-4#UK,NaN,defra rural services helpline;severn edge vete...,"-11.9850187265918,0.374531835205993,12.3595505...",NaN,shropshirestar.com,...,None,None,None,None,None,None,None,None,None,None
2888990,20240131,NaT,"AFFECT#50##3#Washington, District Of Columbia,...",MANMADE_DISASTER_IMPLIED;TAX_FNCACT;TAX_FNCACT...,"2#California, United States#US#USCA#36.17#-119...",NaN,ruth law team;national insurance crime bureau;...,"-1.14942528735632,3.2183908045977,4.3678160919...",1154814764,983thesnake.com,...,None,None,None,None,None,None,None,None,None,None
